# Phase 1 — Hybrid Hierarchical Retrieval on DAPR

DAPR-specific acquisition and preprocessing live in this notebook. Reusable
retrieval, fusion, evaluation, caching, and artifact code come from `dapr_hhr`.
The default synthetic smoke run validates plumbing only; it is not a benchmark.

## 1. Install the reusable hierarchical benchmark package

Upload this notebook to Kaggle and enable Internet. During development use
`main`; for a reported run replace `REPO_REF` with a tested commit SHA.

In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import json
import os
import random
import subprocess
import sys
from collections import defaultdict
from dataclasses import replace
from pathlib import Path
from typing import Any

REPO_REF = "main"
REPO_URL = "https://github.com/ManhTanTran/hierarchical-retrieval-benchmark"
ON_KAGGLE = os.name != "nt" and Path("/kaggle/working").is_dir()


def find_project_root(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "dapr_hhr").is_dir():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if ON_KAGGLE:
    requirements_url = (
        "https://raw.githubusercontent.com/ManhTanTran/"
        f"hierarchical-retrieval-benchmark/{REPO_REF}/requirements-kaggle.txt"
    )
    package_url = f"git+{REPO_URL}.git@{REPO_REF}"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_url])
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "--no-deps",
            package_url,
        ]
    )
elif PROJECT_ROOT is not None:
    source_root = str(PROJECT_ROOT / "src")
    if source_root not in sys.path:
        sys.path.insert(0, source_root)

print({"repo_ref": REPO_REF, "project_root": str(PROJECT_ROOT), "on_kaggle": ON_KAGGLE})

## 2. Central configuration

Edit this cell only. Baseline/full modes use complete selected corpora and can
exceed one Kaggle session. Start with a single dataset and a query sample.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from dapr_hhr import (
    DatasetBundle,
    Document,
    Passage,
    Qrel,
    Query,
    run_phase1_benchmark,
    summarize_dataset,
)
from dapr_hhr.experiments import build_experiment_registry
from dapr_hhr.metrics import compute_ndcg_at_k, compute_recall_at_k
from dapr_hhr.smoke import make_synthetic_bundle

ALL_DAPR_DATASETS = (
    "ms_marco",
    "natural_questions",
    "miracl_en",
    "genomics",
    "conditional_qa",
    "nq_hard",
)
RECOMMENDED_METHODS = (
    "sparse__dense",
    "dense__dense",
    "combined__dense",
    "combined__combined",
)
RUN_MODES = {
    "smoke": {
        "datasets": ("synthetic",),
        "methods": (
            "sparse__sparse",
            "sparse__dense",
            "dense__dense",
            "combined__combined",
        ),
        "query_sample_size": 4,
        "dense_backend": "hashing",
        "document_top_k": 3,
        "passage_top_k": 5,
    },
    "baseline": {
        "datasets": ALL_DAPR_DATASETS,
        "methods": RECOMMENDED_METHODS,
        "query_sample_size": None,
        "dense_backend": "sentence_transformers",
        "document_top_k": 20,
        "passage_top_k": 100,
    },
    "full": {
        "datasets": ALL_DAPR_DATASETS,
        "methods": tuple(build_experiment_registry()),
        "query_sample_size": None,
        "dense_backend": "sentence_transformers",
        "document_top_k": 100,
        "passage_top_k": 100,
    },
}

WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else (PROJECT_ROOT or Path.cwd())
CONFIG = {
    "run_mode": "smoke",  # smoke | baseline | full
    "datasets": None,  # e.g. ("ms_marco",)
    "query_sample_size": None,  # e.g. 100; None uses the mode default
    "hf_repo": "UKPLab/dapr",
    "hf_revision": "67ae3daa13596700976d20605630f5f9db3bd732",
    "hf_cache_dir": WORK_ROOT / "dapr_hf_cache",
    "retrieval_cache_dir": WORK_ROOT / "dapr_hhr_cache",
    "output_dir": WORK_ROOT / "dapr_hhr_outputs",
    "random_seed": 42,
    "fusion": "rrf",
    "dense_model": "intfloat/e5-small-v2",
    "dense_model_revision": "ffb93f3bd4047442299a41ebb6fa998a38507c52",
    "tuning_dataset": "ms_marco",
    "tuning_split": "dev",
    "frozen_parameters": True,
}

mode = dict(RUN_MODES[CONFIG["run_mode"]])
if CONFIG["datasets"] is not None:
    mode["datasets"] = tuple(CONFIG["datasets"])
if CONFIG["query_sample_size"] is not None:
    mode["query_sample_size"] = int(CONFIG["query_sample_size"])
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
print(json.dumps({**CONFIG, **mode}, indent=2, default=str))

## 3. Official DAPR configuration registry and label mapping

In [ ]:
HF_DAPR_SPECS = {
    "ms_marco": {"prefix": "MSMARCO", "query_split": "test"},
    "natural_questions": {"prefix": "NaturalQuestions", "query_split": "test"},
    "miracl_en": {"prefix": "MIRACL", "query_split": "test"},
    "genomics": {"prefix": "Genomics", "query_split": "test"},
    "conditional_qa": {"prefix": "ConditionalQA", "query_split": "test"},
}
NQ_CATEGORY_MAP = {
    "coreference": "CR",
    "main_topic": "MT",
    "multi-hop": "MHR",
    "acronym": "AC",
}


def load_hf_config(config_name: str, split: str):
    from datasets import load_dataset

    print(f"Downloading/loading {config_name}:{split}")
    return load_dataset(
        CONFIG["hf_repo"],
        config_name,
        split=split,
        revision=CONFIG["hf_revision"],
        cache_dir=str(CONFIG["hf_cache_dir"]),
    )


def value(row: dict[str, Any], key: str, default: Any = "") -> Any:
    selected = row.get(key, default)
    return default if selected is None else selected

## 4. Notebook-local DAPR normalization

In [ ]:
def normalize_documents(rows: Any) -> list[Document]:
    documents = []
    for row in rows:
        passage_ids = list(value(row, "passage_ids", []))
        passages = list(value(row, "passages", []))
        if len(passage_ids) != len(passages):
            raise ValueError("DAPR docs contain mismatched passage_ids/passages lengths")
        documents.append(
            Document(
                doc_id=str(value(row, "doc_id")),
                title=str(value(row, "title")),
                text="\n\n".join(map(str, passages)),
            )
        )
    return documents


def normalize_passages(rows: Any) -> list[Passage]:
    passages = []
    for row in rows:
        passages.append(
            Passage(
                passage_id=str(value(row, "_id")),
                doc_id=str(value(row, "doc_id")),
                title=str(value(row, "title")),
                text=str(value(row, "text")),
                paragraph_no=int(value(row, "paragraph_no", 0)),
                metadata={
                    key: row[key] for key in ("is_candidate", "total_paragraphs") if key in row
                },
            )
        )
    return passages


def normalize_queries(rows: Any, dataset_name: str, split: str) -> list[Query]:
    return [
        Query(
            query_id=str(value(row, "_id")),
            text=str(value(row, "text")),
            metadata={"dataset": dataset_name, "split": split},
        )
        for row in rows
    ]


def normalize_qrels(rows: Any) -> list[Qrel]:
    scores: dict[tuple[str, str], float] = {}
    for row in rows:
        key = (str(value(row, "query_id")), str(value(row, "corpus_id")))
        scores[key] = max(float(value(row, "score", 0.0)), scores.get(key, 0.0))
    return [Qrel(query_id, passage_id, score) for (query_id, passage_id), score in scores.items()]


def normalize_nq_hard(rows: Any) -> tuple[list[Query], list[Qrel], dict[str, dict[str, Any]]]:
    queries: dict[str, Query] = {}
    qrel_scores: dict[tuple[str, str], float] = {}
    categories: dict[str, set[str]] = defaultdict(set)
    for row in rows:
        query_id = str(value(row, "query_id"))
        passage_id = str(value(row, "corpus_id"))
        queries.setdefault(
            query_id,
            Query(query_id, str(value(row, "query")), {"dataset": "nq_hard", "split": "test"}),
        )
        key = (query_id, passage_id)
        qrel_scores[key] = max(float(value(row, "score", 0.0)), qrel_scores.get(key, 0.0))
        for category in value(row, "categories", []):
            normalized = NQ_CATEGORY_MAP.get(str(category).strip().lower())
            if normalized:
                categories[query_id].add(normalized)
    qrels = [
        Qrel(query_id, passage_id, score) for (query_id, passage_id), score in qrel_scores.items()
    ]
    metadata = {query_id: {"categories": sorted(labels)} for query_id, labels in categories.items()}
    return list(queries.values()), qrels, metadata

## 5. Sampling, protocol validation, and bundle construction

In [ ]:
def sample_queries(bundle: DatasetBundle, sample_size: int | None, seed: int) -> DatasetBundle:
    if sample_size is None or sample_size >= len(bundle.queries):
        return bundle
    selected = random.Random(seed).sample(bundle.queries, sample_size)
    query_ids = {query.query_id for query in selected}
    sampled = replace(
        bundle,
        queries=selected,
        qrels=[qrel for qrel in bundle.qrels if qrel.query_id in query_ids],
        query_metadata={
            query_id: metadata
            for query_id, metadata in bundle.query_metadata.items()
            if query_id in query_ids
        },
    )
    sampled.validate()
    return sampled


def validate_zero_shot_protocol() -> None:
    if CONFIG["tuning_dataset"] != "ms_marco" or CONFIG["tuning_split"] not in {"train", "dev"}:
        raise ValueError("Parameter selection is limited to MS MARCO train/dev")
    if CONFIG["run_mode"] != "smoke" and not CONFIG["frozen_parameters"]:
        raise ValueError("Real evaluation requires frozen parameters")


def load_dapr_bundle(dataset_name: str) -> DatasetBundle:
    if dataset_name == "nq_hard":
        prefix = "NaturalQuestions"
        documents = normalize_documents(load_hf_config(f"{prefix}-docs", "test"))
        passages = normalize_passages(load_hf_config(f"{prefix}-corpus", "test"))
        queries, qrels, metadata = normalize_nq_hard(load_hf_config("nq-hard", "test"))
    else:
        spec = HF_DAPR_SPECS[dataset_name]
        prefix = spec["prefix"]
        split = spec["query_split"]
        documents = normalize_documents(load_hf_config(f"{prefix}-docs", "test"))
        passages = normalize_passages(load_hf_config(f"{prefix}-corpus", "test"))
        queries = normalize_queries(load_hf_config(f"{prefix}-queries", split), dataset_name, split)
        qrels = normalize_qrels(load_hf_config(f"{prefix}-qrels", split))
        metadata = {}
    bundle = DatasetBundle(
        documents=documents,
        passages=passages,
        queries=queries,
        qrels=qrels,
        name=dataset_name,
        query_metadata=metadata,
        provenance={
            "source": CONFIG["hf_repo"],
            "revision": CONFIG["hf_revision"],
            "query_split": "test",
        },
    )
    bundle.validate()
    return sample_queries(bundle, mode["query_sample_size"], CONFIG["random_seed"])


validate_zero_shot_protocol()

## 6. Normalizer smoke assertions

In [ ]:
mini_docs = [{"doc_id": "d", "title": "T", "passage_ids": ["p"], "passages": ["body"]}]
mini_corpus = [{"_id": "p", "doc_id": "d", "title": "T", "text": "body", "paragraph_no": 0}]
mini_nq = [
    {
        "query_id": "q",
        "corpus_id": "p",
        "score": 1,
        "query": "question",
        "categories": ["coreference"],
    },
    {
        "query_id": "q",
        "corpus_id": "p2",
        "score": 1,
        "query": "question",
        "categories": ["acronym", "coreference"],
    },
]
assert normalize_documents(mini_docs)[0].text == "body"
assert normalize_passages(mini_corpus)[0].passage_id == "p"
_, mini_qrels, mini_metadata = normalize_nq_hard(mini_nq)
assert len(mini_qrels) == 2
assert set(mini_metadata["q"]["categories"]) == {"AC", "CR"}
print("Official-shape normalizer checks passed.")

## 7. Load and audit selected datasets

In [ ]:
bundles = [
    make_synthetic_bundle() if name == "synthetic" else load_dapr_bundle(name)
    for name in mode["datasets"]
]
for bundle in bundles:
    bundle.validate()
dataset_audit = pd.DataFrame([summarize_dataset(bundle) for bundle in bundles])
display(dataset_audit)

## 8. Experiment registry

In [ ]:
registry = build_experiment_registry()
assert len(registry) == 9
assert all(method in registry for method in mode["methods"])
display(pd.DataFrame([vars(experiment) for experiment in registry.values()]))

## 9. Metric contract validation

In [ ]:
relevance = {"high": 2, "medium": 1, "none": 0}
actual = compute_ndcg_at_k(["medium", "high", "none"], relevance, 10)
expected = (1.0 + 3.0 / np.log2(3)) / (3.0 + 1.0 / np.log2(3))
assert np.isclose(actual, expected)
assert np.isclose(compute_recall_at_k(["none", "high"], relevance, 2), 0.5)
print("Metric checks passed.")

## 10. Execute reusable Phase 1 workflow

In [ ]:
reports = []
for bundle in bundles:
    report = run_phase1_benchmark(
        bundle,
        run_mode=CONFIG["run_mode"],
        fusion=CONFIG["fusion"],
        dense_backend=mode["dense_backend"],
        dense_model=CONFIG["dense_model"],
        dense_model_revision=CONFIG["dense_model_revision"],
        document_top_k=mode["document_top_k"],
        passage_top_k=mode["passage_top_k"],
        experiment_names=mode["methods"],
        cache_dir=CONFIG["retrieval_cache_dir"],
        output_dir=CONFIG["output_dir"],
        query_metadata=bundle.query_metadata,
    )
    reports.append(report)
print(f"Completed {sum(len(report.leaderboard) for report in reports)} dataset/method runs.")

## 11. Effectiveness and latency results

In [ ]:
leaderboard = pd.concat(
    [
        report.leaderboard.assign(dataset=bundle.name)
        for bundle, report in zip(bundles, reports, strict=True)
    ],
    ignore_index=True,
)
display(leaderboard.sort_values(["dataset", "passage_ndcg@10"], ascending=[True, False]))

## 12. Dataset/domain comparison

In [ ]:
domain_comparison = leaderboard.pivot_table(
    index="experiment",
    columns="dataset",
    values=["passage_ndcg@10", "passage_recall@100", "latency_ms"],
)
display(domain_comparison)

## 13. NQ-hard diagnostics

In [ ]:
grouped_rows = []
for bundle, report in zip(bundles, reports, strict=True):
    for category, metrics in report.grouped_metrics.items():
        grouped_rows.append({"dataset": bundle.name, "category": category, **metrics})
grouped_metrics = pd.DataFrame(grouped_rows)
display(grouped_metrics if len(grouped_metrics) else pd.DataFrame({"status": ["not_run"]}))

## 14. Artifact locations and conclusion

In [ ]:
artifact_table = pd.DataFrame(
    [
        {
            "dataset": bundle.name,
            "best_experiment": report.best_experiment,
            "artifact_root": str(report.artifact_root),
            "download_zip": str(report.archive_path),
        }
        for bundle, report in zip(bundles, reports, strict=True)
    ]
)
display(artifact_table)
if CONFIG["run_mode"] == "smoke":
    print("Synthetic smoke run complete; do not report these values as DAPR results.")
else:
    measured = (
        leaderboard.groupby("experiment")["passage_ndcg@10"].mean().sort_values(ascending=False)
    )
    print(f"Measured leader: {measured.index[0]} ({measured.iloc[0]:.4f} mean passage nDCG@10)")